# Amazon Bedrock AgentCore Runtime의 Distributed Multi-agent Solution

## 개요

이 튜토리얼에서는 서로 다른 Agentic Framework로 구축한 각 에이전트를 개별 Bedrock AgentCore Runtime에 독립적으로 호스팅하는 방법을 알아봅니다. 그런 다음 distributed multi-agent solution을 위해 에이전트 간 통신을 활성화합니다. 

이 예제에서는 다음 에이전트를 생성합니다.
1. programming 및 기술 troubleshooting 질문에 전문적으로 답변하는 technical agent(`tech_agent`)
2. 회사 복리후생을 전문으로 하는 HR agent(`hr_agent`)
3. 질문을 technical agent 또는 HR agent로 routing하는 orchestrator agent(`orchestrator_agent`)

이 세 에이전트를 결합하면 사용자 질문을 적절한 subagent로 routing할 수 있는 supervisor 기반 multi-agent 구성이 만들어집니다. 이 시스템은 회사 직원이 가질 수 있는 다양한 질문에 답변할 수 있습니다.


### 튜토리얼 세부 정보


| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                            |
| Agent 유형          | Multi-Agent(Supervisor가 agent를 tool로 호출)                                                                              |
| Agentic Framework   | Strands Agents 및 LangGraph                                                                |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                        |
| 튜토리얼 구성 요소  | AgentCore Runtime에 agent 호스팅 및 multi-agent collaboration 활성화     |
| 튜토리얼 분야       | 산업 공통                                                                        |
| 예제 난이도         | 중급                                                                               |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 boto3                                      |

### 튜토리얼 아키텍처

이 튜토리얼에서는 세 에이전트를 Bedrock AgentCore Runtime에 배포하는 방법을 설명합니다. Orchestrator와 Tech agent에는 Strands Agent를, HR agent에는 LangGraph agent를 사용합니다. 각 에이전트를 자체 AgentCore Runtime에 배포하여 서로 다른 agent framework가 혼합된 multi-agent system을 구성하는 방법을 간단한 에이전트로 보여줍니다.

![아키텍처](./architecture.png)


### 튜토리얼 주요 기능

* 여러 Agent를 Amazon Bedrock AgentCore Runtime에 호스팅
* 각 agent가 독립적으로 호스팅되는 Multi-agent solution 생성


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents
* LangGraph

In [ ]:
!uv add -r requirements.txt --active

In [ ]:
import os

# environment variable 설정
os.environ["AWS_DEFAULT_REGION"] = "us-west-2"

## Agent 생성

먼저 각 에이전트에 별도의 IAM role 세 개를 생성합니다. 이를 통해 다른 에이전트와 독립적으로 각 에이전트에 least privilege permission을 정의할 수 있습니다.

In [ ]:
from utils import create_agentcore_role

tech_agent_name = "tech_agent"
tech_agent_iam_role = create_agentcore_role(agent_name=tech_agent_name, region=os.getenv("AWS_DEFAULT_REGION"))
tech_agent_role_arn = tech_agent_iam_role["Role"]["Arn"]
tech_agent_role_name = tech_agent_iam_role["Role"]["RoleName"]
print(tech_agent_role_arn)
print(tech_agent_role_name)

hr_agent_name = "hr_agent"
hr_agent_iam_role = create_agentcore_role(agent_name=hr_agent_name, region=os.getenv("AWS_DEFAULT_REGION"))
hr_agent_role_arn = hr_agent_iam_role["Role"]["Arn"]
hr_agent_role_name = hr_agent_iam_role["Role"]["RoleName"]
print(hr_agent_role_arn)
print(hr_agent_role_name)

orchestrator_agent_name = "orchestrator_agent"
orchestrator_iam_role = create_agentcore_role(
    agent_name=orchestrator_agent_name, region=os.getenv("AWS_DEFAULT_REGION")
)
orchestrator_role_arn = orchestrator_iam_role["Role"]["Arn"]
orchestrator_role_name = orchestrator_iam_role["Role"]["RoleName"]
print(orchestrator_role_arn)
print(orchestrator_role_name)

### Helper 함수

* `configure_runtime` helper 함수는 각 에이전트의 Runtime 구성을 설정하는 데 사용됩니다. 이 예제에서는 starter toolkit을 사용하여 entrypoint, 방금 생성한 execution role, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 또한 시작할 때 Amazon ECR repository를 자동으로 생성하도록 starter toolkit을 구성합니다.
* `check_status` helper 함수는 AWS 계정에 배포된 각 Runtime을 확인하여 성공적으로 생성되었고 에이전트를 사용할 준비가 되었는지 검증합니다.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import time


def configure_runtime(agent_name, agentcore_iam_role, python_file_name):
    boto_session = Session(region_name=os.getenv("AWS_DEFAULT_REGION"))
    region = boto_session.region_name

    agentcore_runtime = Runtime()

    response = agentcore_runtime.configure(
        entrypoint=python_file_name,
        execution_role=agentcore_iam_role["Role"]["Arn"],
        auto_create_ecr=True,
        requirements_file="requirements.txt",
        region=region,
        agent_name=agent_name,
    )
    return response, agentcore_runtime


def check_status(agent_runtime):
    status_response = agent_runtime.status()
    status = status_response.endpoint["status"]
    end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
    while status not in end_status:
        time.sleep(10)
        status_response = agent_runtime.status()
        status = status_response.endpoint["status"]
        print(status)
    return status

In [ ]:
# 현재 working directory를 tech_agent 폴더로 설정
import os

os.chdir("./tech_agent")
print(os.getcwd())

### Tech Support Agent 생성(Strands Agents)

Strands와 Amazon Bedrock 모델을 사용하는 Tech Support Agent부터 시작합니다. 다음 셀을 실행하면 agent별 logic이 포함된 `tech_agent.py` 파일이 `./tech_agent` 디렉터리에 생성됩니다. 

app은 `BedrockAgentCoreApp()`으로 정의되고 invocation 함수 `strands_agent_bedrock`에는 `@app.entrypoint` decorator가 적용되며 파일 끝에는 `app.run()` 명령이 있다는 점에 유의하세요.

In [ ]:
%%writefile tech_agent.py

from strands import Agent, tool
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel

app = BedrockAgentCoreApp()

model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    system_prompt="You're a helpful tech support assistant, you can help user questions on tech troubleshooting and programming"
)

@app.entrypoint
def strands_agent_bedrock(payload):
    """
    payload로 에이전트를 호출합니다.
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

#### Agent 시작

먼저 configure_runtime helper 함수를 사용하여 agent 배포에 필요한 .bedrock_agentcore.yaml, .dockerignore, Dockerfile을 생성합니다. 그런 다음 Runtime에서 .launch()를 호출하여 image를 ECR에 push하고 AWS 환경에 AgentCore Runtime을 생성합니다. 


In [ ]:
_, tech_agent_runtime = configure_runtime("tech_agent", tech_agent_iam_role, "tech_agent.py")
tech_launch_result = tech_agent_runtime.launch()
tech_agent_id = tech_launch_result.agent_id
tech_agent_arn = tech_launch_result.agent_arn

print(tech_agent_arn)

#### Tech Agent ARN을 Parameter Store에 저장 

Tech Agent의 AgentCore Runtime ARN을 지속적으로 저장하고 조회할 수 있는 간단한 agent registry를 생성합니다.

In [ ]:
import boto3

ssm = boto3.client("ssm")
ssm.put_parameter(Name="/agents/tech_agent_arn", Value=tech_agent_arn, Type="String", Overwrite=True)

#### Agent 테스트

에이전트를 테스트하려면 먼저 Tech Agent AgentCore Runtime의 상태를 확인하여 사용할 준비가 되었는지 검증합니다.\
tech_agent_runtime에서 `.invoke()`를 사용하여 agent runtime이 구성되었고 예상대로 작동하는지 검증합니다.

In [ ]:
status = check_status(tech_agent_runtime)
print(status)

In [ ]:
invoke_response = tech_agent_runtime.invoke({"prompt": "shortcut to minimize windows in Mac, in 1 sentence"})
invoke_response

### HR Agent 생성(Langraph Agents)

비슷한 과정으로 HR Agent를 생성합니다. 이번에는 기본 agent logic이 LangGraph로 구축된 것을 확인할 수 있습니다. Agent Framework 변경은 AgentCore Runtime 구성 방식에 영향을 주지 않습니다. 다음 셀을 실행하면 `./hr_agent` 디렉터리에 `hr_agent.py` 파일이 생성됩니다.

Tech Support Agent와 마찬가지로 app을 `BedrockAgentCoreApp()`으로 정의하고 invocation 함수 langgraph_bedrock에 @app.entrypoint decorator를 적용하며 파일 끝에 `app.run()` 명령을 둡니다. 

In [ ]:
# 현재 working directory를 hr_agent 폴더로 설정
import os

os.chdir("../hr_agent")
print(os.getcwd())

In [ ]:
%%writefile hr_agent.py
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from bedrock_agentcore.runtime import BedrockAgentCoreApp
import argparse
import json
import operator
import math

app = BedrockAgentCoreApp()

@tool
def get_vacation_info():
    """Get remaining vacation days balance for the current year"""  # dummy 구현
    return "you have 12 days off remaining this year"

# 수동 LangGraph 구성으로 agent 정의
def create_agent():
    """LangGraph 에이전트를 생성하고 구성합니다."""
    from langchain_aws import ChatBedrock
    
    # LLM 초기화(필요에 따라 model 및 parameter 조정)
    llm = ChatBedrock(
        model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # 또는 선호하는 model
        model_kwargs={"temperature": 0.1}
    )
    
    # tool을 LLM에 binding
    tools = [get_vacation_info]
    llm_with_tools = llm.bind_tools(tools)
    
    # system message 설정
    system_message = f"""You're a helpful hr support assistant, you can answers user questions on vacations and benefits. 
    Here are the primary company benefits
    - Comprehensive health insurance with 100% premium coverage for employees and 75% for dependents
    - Flexible PTO policy with 20 days paid vacation annually, plus 5 sick days
    - 401(k) plan with 6% company matching and immediate vesting
    - Monthly wellness stipend of $100 for gym memberships or fitness activities

    For additional HR information instruct the user to call to 1-800-ASKHR"""
    
    # chatbot node 정의
    def chatbot(state: MessagesState):
        # 아직 없는 경우 system message 추가
        messages = state["messages"]
        if not messages or not isinstance(messages[0], SystemMessage):
            messages = [SystemMessage(content=system_message)] + messages
        
        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}
    
    # graph 생성
    graph_builder = StateGraph(MessagesState)
    
    # node 추가
    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", ToolNode(tools))
    
    # edge 추가
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")
    
    # entry point 설정
    graph_builder.set_entry_point("chatbot")
    
    # graph compile 수행
    return graph_builder.compile()

# agent 초기화
agent = create_agent()

@app.entrypoint
def langgraph_bedrock(payload):
    """
    payload로 에이전트를 호출합니다.
    """
    user_input = payload.get("prompt")
    
    # LangGraph에서 예상하는 형식으로 input 생성
    response = agent.invoke({"messages": [HumanMessage(content=user_input)]})
    
    # 최종 message content 추출
    return response["messages"][-1].content

if __name__ == "__main__":
    app.run()

#### Agent 시작

다시 configure_runtime helper 함수를 사용하여 agent 배포에 필요한 .bedrock_agentcore.yaml, .dockerignore, Dockerfile을 생성합니다. 그런 다음 HR agent runtime에서 .launch()를 호출하여 image를 ECR에 push하고 AWS 환경에 AgentCore Runtime을 생성합니다. 

In [ ]:
_, hr_agentcore_runtime = configure_runtime("hr_agent", hr_agent_iam_role, "hr_agent.py")
hr_launch_result = hr_agentcore_runtime.launch()
hr_agent_id = hr_launch_result.agent_id
hr_agent_arn = hr_launch_result.agent_arn

print(hr_agent_arn)

#### HR Agent ARN을 Parameter Store에 저장

HR Agent의 AgentCore Runtime ARN을 지속적으로 저장하고 조회할 수 있도록 간단한 agent registry를 확장합니다.

In [ ]:
import boto3

ssm = boto3.client("ssm")
ssm.put_parameter(Name="/agents/hr_agent_arn", Value=hr_agent_arn, Type="String", Overwrite=True)

#### Agent 테스트

HR Agent AgentCore Runtime의 상태를 확인하여 사용할 준비가 되었는지 검증합니다.\
hr_agentcore_runtime에서 `.invoke()`를 사용하여 AgentCore Runtime이 구성되었고 예상대로 작동하는지 검증합니다.

In [ ]:
status = check_status(hr_agentcore_runtime)
status

In [ ]:
# agent 테스트
invoke_response = hr_agentcore_runtime.invoke({"prompt": "How many vacation days I have left?"})
invoke_response

### Orchestrator Agent 생성(Strands Agents)

세 번째 에이전트인 orchestrator에는 다시 Strands를 Agent framework로 사용합니다. 에이전트를 생성하기 전에 앞에서 생성한 AgentCore Runtime execution role을 업데이트하여 Tech Support Agent와 HR Agent를 호출할 권한을 부여해야 합니다.

아래 `update_orchestrator_permissions` 함수는 subagent ARN과 Parameter Store에 등록된 agent ARN을 받아 orchestrator agent에 subagent의 DEFAULT runtime endpoint를 호출할 권한을 부여합니다. 또한 Orchestrator Agent가 Parameter Store에서 Agent ARN을 가져올 권한도 부여합니다.

In [ ]:
# 필요한 subagent를 호출할 권한이 있도록 orchestrator AgentCore execution role 업데이트
# orchestrator에는 Parameter Store에서 subagent ARN을 가져올 권한도 필요함
import json

# Parameter Store에서 runtime ARN 가져오기
ssm = boto3.client("ssm")
response = ssm.get_parameter(Name="/agents/tech_agent_arn")
tech_agent_arn = response["Parameter"]["Value"]
tech_agent_parameter_arn = response["Parameter"]["ARN"]

ssm = boto3.client("ssm")
response = ssm.get_parameter(Name="/agents/hr_agent_arn")
hr_agent_arn = response["Parameter"]["Value"]
hr_agent_parameter_arn = response["Parameter"]["ARN"]


def update_orchestrator_permissions(sub_agent_arns: list, sub_agent_parameter_arns: list, orchestrator_name: str):
    iam_client = boto3.client("iam")
    orchestrator_permissions = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": ["bedrock-agentcore:InvokeAgentRuntime"],
                "Resource": [sub_agent_arn + "/runtime-endpoint/DEFAULT" for sub_agent_arn in sub_agent_arns]
                + [sub_agent_arn for sub_agent_arn in sub_agent_arns],
            },
            {
                "Effect": "Allow",
                "Action": ["ssm:GetParameter"],
                "Resource": [sub_agent_parameter_arn for sub_agent_parameter_arn in sub_agent_parameter_arns],
            },
        ],
    }

    rsp = iam_client.put_role_policy(
        RoleName=orchestrator_name,
        PolicyName="subagent_permissions-new",
        PolicyDocument=json.dumps(orchestrator_permissions),
    )
    return rsp


rsp = update_orchestrator_permissions(
    [tech_agent_arn, hr_agent_arn],
    [tech_agent_parameter_arn, hr_agent_parameter_arn],
    orchestrator_role_name,
)
print(rsp)

In [ ]:
# 현재 working directory를 orchestrator_agent 폴더로 설정
import os

os.chdir("../orchestrator_agent")
print(os.getcwd())

다음 셀을 실행하면 agent별 logic이 포함된 `orchestrator_agent.py` 파일이 `./orchestrator_agent` 디렉터리에 생성됩니다. orchestrator agent는 1. `call_tech_agent`, 2. `call_HR_agent`의 두 tool을 사용할 수 있습니다. 두 tool 모두 `./orchestrator_agent` 폴더의 `invoke_agent_utils.py` 파일에 미리 정의된 invoke_agent_utils 함수를 사용합니다. subagent는 boto3의 invoke_agent_runtime action을 통해 tool로 호출됩니다.

이처럼 복잡한 agent 설정에서도 Bedrock AgentCore Runtime 구성은 동일합니다. app은 `BedrockAgentCoreApp()`으로 정의되고 invocation 함수 `strands_agent_bedrock_streaming`에는 `@app.entrypoint` decorator가 적용되며 파일 끝에는 `app.run()` 명령이 있다는 점에 유의하세요.

In [ ]:
%%writefile orchestrator_agent.py

import argparse
import json
import boto3
import logging

from strands import Agent, tool
from strands_tools import calculator 
from strands.models import BedrockModel

from bedrock_agentcore.runtime import BedrockAgentCoreApp

from invoke_agent_utils import invoke_agent_with_boto3

logger = logging.getLogger(__name__)

app = BedrockAgentCoreApp()

def get_agent_arn(agent_name: str) -> str:
    """
    Parameter Store에서 에이전트 ARN을 조회합니다.
    """
    try:
        ssm = boto3.client('ssm')
        response = ssm.get_parameter(
            Name=f'/agents/{agent_name}_arn'
        )
        return response['Parameter']['Value']
    except Exception as err:
        print(err)
        raise err

@tool
def call_tech_agent(user_query):
    """ call the tech agent """ 
    # print("Calling tech agent")
    try:
        tech_agent_arn = get_agent_arn ("tech_agent")
        result = invoke_agent_with_boto3(tech_agent_arn, user_query=user_query)
    except Exception as e:
        result = str(e)
        logger.exception("Exception calling tech agent: ")
    return result

@tool
def call_HR_agent(user_query):
    """ Get the HR agent """ 
    print("Calling HR agent")
    try:
        hr_agent_arn = get_agent_arn("hr_agent")
        print(hr_agent_arn)
        result = invoke_agent_with_boto3(hr_agent_arn, user_query=user_query)
    except Exception as e:
        result = str(e)
        logger.error(f"Exception calling hr agent: {e}", exc_info=True)
    return result


model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    system_prompt="You're a helpful assistant, your role is to understand user questions and delegate to the appropriate specialized agent, you have tools to call the tech and HR agents",
    tools=[call_tech_agent, call_HR_agent]
)

def parse_event(event):
    """
    에이전트의 스트리밍 이벤트를 파싱하고 형식화된 출력을 반환합니다.
    """
    # 표시할 필요가 없는 event 건너뛰기
    if any(key in event for key in ['init_event_loop', 'start', 'start_event_loop']):
        return ""
    
    # supervisor의 text chunk
    if 'data' in event and isinstance(event['data'], str):
        return event['data'] 
    
    
    # assistant의 text message 처리
    if 'event' in event:
        event_data = event['event']
        
        # tool 사용 시작
        if 'contentBlockStart' in event_data and 'start' in event_data['contentBlockStart']:
            if 'toolUse' in event_data['contentBlockStart']['start']:
                tool_info = event_data['contentBlockStart']['start']['toolUse']
                return f"\n\n[Executing: {tool_info['name']}]\n\n"        

    return ""

@app.entrypoint
async def strands_agent_bedrock_streaming(payload):
    """
    스트리밍 기능으로 에이전트를 호출합니다.
    이 함수는 비동기 제너레이터를 사용해 AgentCore Runtime에서
    스트리밍 응답을 구현하는 방법을 보여 줍니다.
    """
    user_input = payload.get("prompt")
    #print("User input:", user_input)
    
    try:
        # 사용 가능해지는 즉시 각 chunk streaming
        async for event in agent.stream_async(user_input):
            text = parse_event(event)
            if text:  # 비어 있지 않은 response만 반환
                yield text
                
            #if "data" in event:
            #    yield event["data"]
            
    except Exception as e:
        # streaming context에서 오류를 원활하게 처리
        error_response = {"error": str(e), "type": "stream_error"}
        print(f"Streaming error: {error_response}")
        yield error_response


if __name__ == "__main__":
    app.run()

#### Agent 시작

다시 configure_runtime helper 함수를 사용하여 agent 배포에 필요한 .bedrock_agentcore.yaml, .dockerignore, Dockerfile을 생성합니다. 그런 다음 Orchestrator Agent runtime에서 .launch()를 호출하여 image를 ECR에 push하고 AWS 환경에 AgentCore Runtime을 생성합니다. 

In [ ]:
_, orchestrator_agentcore_runtime = configure_runtime(
    "orchestrator_agent", orchestrator_iam_role, "orchestrator_agent.py"
)
orchestrator_launch_result = orchestrator_agentcore_runtime.launch()

#### Agent 테스트

Orchestrator Agent AgentCore Runtime의 상태를 확인하여 사용할 준비가 되었는지 검증합니다.\

이번에는 utils의 `invoke_agent_with_boto3` 함수를 사용하여 Orchestrator Agent를 테스트할 수 있습니다. Tech Support Agent와 HR Agent를 모두 호출하게 될 질문을 orchestrator에 해봅니다.

In [ ]:
status = check_status(orchestrator_agentcore_runtime)
print(status)

from invoke_agent_utils import invoke_agent_with_boto3


result = invoke_agent_with_boto3(
    orchestrator_launch_result.agent_arn,
    "tell me about my benefits, also tell me how to connect a bluetooth mouse to my mac",
)

## 리소스 정리(선택 사항)

이제 생성된 AgentCore Runtime을 정리합니다.

In [ ]:
print(
    orchestrator_launch_result.ecr_uri,
    orchestrator_launch_result.agent_id,
    orchestrator_launch_result.ecr_uri.split("/")[1],
)
print(
    hr_launch_result.ecr_uri,
    hr_launch_result.agent_id,
    hr_launch_result.ecr_uri.split("/")[1],
)
print(
    tech_launch_result.ecr_uri,
    tech_launch_result.agent_id,
    tech_launch_result.ecr_uri.split("/")[1],
)

In [ ]:
def clean_up_agent_runtimes(launch_result):
    agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=os.getenv("AWS_DEFAULT_REGION"))
    ecr_client = boto3.client("ecr", region_name=os.getenv("AWS_DEFAULT_REGION"))
    agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )

    response = ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)

    return response


def delete_iam_roles(agentcore_iam_role):
    iam_client = boto3.client("iam")
    policies = iam_client.list_role_policies(RoleName=agentcore_iam_role["Role"]["RoleName"], MaxItems=100)

    for policy_name in policies["PolicyNames"]:
        iam_client.delete_role_policy(RoleName=agentcore_iam_role["Role"]["RoleName"], PolicyName=policy_name)
    iam_response = iam_client.delete_role(RoleName=agentcore_iam_role["Role"]["RoleName"])
    return iam_response

In [ ]:
print(clean_up_agent_runtimes(hr_launch_result))
print(clean_up_agent_runtimes(tech_launch_result))
print(clean_up_agent_runtimes(orchestrator_launch_result))
print(delete_iam_roles(tech_agent_iam_role))
print(delete_iam_roles(hr_agent_iam_role))
print(delete_iam_roles(orchestrator_iam_role))

# 축하합니다!